### Inicializando las librerias a ocupar ###

In [47]:
import pyarrow.parquet as pq
import gcsfs
import pandas as pd
from pandas_gbq import to_gbq
import bigframes.pandas as bpd
import json
from google.cloud import storage


### Librerias a instalar ###

In [48]:
#pip install bigframes
#pip install pandas-gbq
#pip install google-cloud-bigquery
#pip install  --upgrade bigframes

### Información BIG QUERY

In [49]:
PROJECT_ID = 'adsac-455509'
LOCATION = "southamerica-east1" 

### insertando opciones de BigQuery DataFrames

In [50]:
bpd.options.bigquery.project = PROJECT_ID
bpd.options.bigquery.location = LOCATION
bpd.options.display.progress_bar = None

## Starting with Business picklet ##

In [51]:
# Create a GCS file system object
fs = gcsfs.GCSFileSystem(project=PROJECT_ID)

# Path to your Parquet file in GCS
file_path = 'gs://adsac/Yelp/business.pkl'
df_business = pd.read_pickle(file_path)

In [52]:
#Eliminando duplicados
df_business = df_business.loc[:, ~df_business.columns.duplicated()]

#Actualizando tipos
df_business['attributes'] = df_business['attributes'].astype(str)
df_business['hours'] = df_business['hours'].astype(str)

In [53]:
#Eliminando nulos
df_business = df_business[df_business['business_id'].isnull() == False]
df_business = df_business[df_business['latitude'].isnull() == False]
df_business = df_business[df_business['longitude'].isnull() == False]
df_business = df_business[df_business['stars'].isnull() == False]
df_business = df_business[df_business['review_count'].isnull() == False]
df_business = df_business[df_business['is_open'].isnull() == False]


In [54]:
#Insertar en la tabla curada
destination_table = 'adsac-455509.Curated.Business'

# Upload DataFrame
to_gbq(df_business, destination_table, project_id=PROJECT_ID, if_exists='replace')  # Use 'append' if you want to add rows

100%|██████████| 1/1 [00:00<?, ?it/s]


## Para leer la tabla curada en big query

In [55]:
# This is how you read a BigQuery table
#df_business_curated = bpd.read_gbq("adsac-455509.Curated.Business")
#df_business_curated.peek()

### Leyendo el archivo tipo Parquet ###

In [56]:
# Create a GCS file system object
fs = gcsfs.GCSFileSystem(project=PROJECT_ID)

# Path to your Parquet file in GCS
file_path = 'gs://adsac/Yelp/user.parquet'

# Open the file and read it
with fs.open(file_path) as f:
    parquet_file = pq.ParquetFile(f)
    df_user = parquet_file.read().to_pandas()

print(df_user.head())

KeyboardInterrupt: 

## Borrando columnas y filas inecesarias 

In [ ]:
#Eliminando archivos 
df_user = df_user[df_user['review_count'] > 0]
df_user = df_user.drop(columns=['compliment_more', 'compliment_profile', 'compliment_cute', 'compliment_list', 'compliment_note', 'compliment_plain', 'compliment_cool'])
df_user = df_user.drop(columns=['compliment_hot', 'compliment_funny', 'compliment_writer', 'compliment_photos'])


### Insertando en BigQuery

In [ ]:
#Insertar en la tabla curada
destination_table = 'adsac-455509.Curated.Usuario'

# Upload DataFrame
to_gbq(df_user, destination_table, project_id=PROJECT_ID, if_exists='replace')  # Use 'append' if you want to add rows

### Para leer la tabla curada en big query

In [ ]:
# This is how you read a BigQuery table
#df_user_curated = bpd.read_gbq("adsac-455509.Curated.Usuario")
#df_user_curated.peek()

,user_id,name,review_count,yelping_since,useful,funny,cool,elite,friends,fans,average_stars
798403,cX6CwuUFV_B5maGj6zlXkw,Sei,1,2018-09-18 22:39:19,0,0,0,,None,0,1.0
798534,9iJcS_yAf4kDGwyrRGi_5A,Teddy,1,2014-06-22 00:52:31,0,0,0,,None,0,5.0
798565,AAxw_r1S3SBlydZfM0AOoQ,Liz,3,2014-10-08 15:04:10,2,0,0,,None,0,5.0
798263,RVcoVbbrQP9me1brqHeJ2Q,Kenneth,4,2012-06-12 17:37:37,3,1,2,,"OPc_47Iuzb8doUQbhiKuSA, lUr1OnGz8n1Q3S0l9tdMUQ...",0,3.75
798594,Pxvu4OU8vuyVJAIkTA37gg,Thuong,217,2017-01-01 00:02:56,166,41,90,"2019,20,20,2021","bwF0z1y_srWijUwmi7_kKQ, 6_i6eYOznkIZty18j5vuZg...",9,3.47


### CHECK IN JSON

In [ ]:
# This is how you read a BigQuery table
df_checkin = bpd.read_gbq("adsac-455509.Staging.CheckIn")
df_checkin_pandas = df_checkin.to_pandas()
df_checkin_pandas = df_checkin_pandas.drop_duplicates()

In [ ]:
#Eliminando los nulls
df_checkin_pandas = df_checkin_pandas[df_checkin_pandas['business_id'].isnull() == False]
df_checkin_pandas = df_checkin_pandas[df_checkin_pandas['date'].isnull() == False]

#### Leer tabla curada

In [ ]:
#This is how you read a BigQuery table
df_checkin_pandas = bpd.read_gbq("adsac-455509.Curated.CheckIn")
df_checkin_pandas.peek()

,date,business_id
95009,"2010-11-27 04:05:50, 2010-12-20 04:34:26, 2011...",d7W1Fi6uRYUhDT_uW83d7w
94669,2011-04-30 15:43:55,SezTh5uY5IY1OZEracWmMw
94990,"2019-11-16 19:48:55, 2020-04-07 16:50:19, 2020...",a6q_KuJs285A4m3YnBVLjQ
95018,"2016-03-02 01:00:43, 2016-03-02 20:48:34, 2016...",SOsjW1JARmtHUFtpFlp8rw
94567,"2014-09-17 19:31:02, 2014-09-22 20:45:14, 2014...",pHmGdzi7B2NpkWR1YKtVbg


#### Agregando la tabla curada a Google Storage

In [ ]:
destination_table = 'adsac-455509.Curated.CheckIn'

# Upload DataFrame
to_gbq(df_checkin_pandas, destination_table, project_id=PROJECT_ID, if_exists='replace')  # Use 'append' if you want to add rows

### Reviews

In [ ]:
# This is how you read a BigQuery table
df_review = bpd.read_gbq("adsac-455509.Staging.Review")
df_review_pandas = df_review.to_pandas()
df_review_pandas = df_review_pandas.drop_duplicates()

In [ ]:
df_review_pandas = df_review_pandas[df_review_pandas['business_id'].isnull() == False]
df_review_pandas = df_review_pandas[df_review_pandas['user_id'].isnull() == False]
df_review_pandas = df_review_pandas[df_review_pandas['text'].isnull() == False]

In [ ]:
df_review_pandas = df_review_pandas.drop(columns=['cool', 'useful','funny'])

In [ ]:
destination_table = 'adsac-455509.Curated.Review'

# Upload DataFrame
to_gbq(df_review_pandas, destination_table, project_id=PROJECT_ID, if_exists='replace')  # Use 'append' if you want to add rows

### Obtener tabla Curada

In [ ]:
#This is how you read a BigQuery table
df_checkin_pandas = bpd.read_gbq("adsac-455509.Curated.Review")
df_checkin_pandas.peek()

,text,stars,date,review_id,business_id,user_id
1726811,"First off, I loved the atmosphere! We walked i...",3.0,2011-10-09 18:33:37+00:00,mN55rTSsmWPpNgW-4OsF1Q,CxzaEZX7Zu8xHBqOBlzqFQ,ieddiannWKFvVmBGS778Vw
1726343,I always come stop by for donuts and sticky bu...,3.0,2015-11-27 02:41:31+00:00,BKPdccRAr8zeDfJmDQubeA,KCVv4CFsiWZnIMaLdGteuQ,UqyP8F6MRg5p2ZAr5CXz1Q
1726598,"Ate lunch here the other day, while in town fo...",4.0,2014-07-24 12:55:49+00:00,HiWj6Y0247zfIK25mjzEwA,WrgdQF8kzvONbZctSPlF4A,Snk_n5DNp_mHuWDVH_nMmg
1726163,Best West Indian (Southern Caribbean) restaura...,5.0,2017-11-18 16:33:38+00:00,5AHrAaUrgtt3wuRHU1of9A,ZuwJdk2g6XGRWV94TE50Cg,9lToW9IE_KmWNE8aOzXRzA
1726241,Took my truck in for an oil leak . Had to retu...,5.0,2018-02-19 19:02:27+00:00,ogHcaYdxmsaSdiFp-euP3A,oAIGgWkdVz9CL5J_2RhznQ,EBC3iaiDs10Lcc0SsU3p0w


### Sitios

In [ ]:
from google.cloud import storage

client = storage.Client.from_service_account_json('adsac-455509-0d5538a0d624.json')
bucket = client.get_bucket('adsac')
blob = bucket.blob('Google/1.json')
content = blob.download_as_text()


In [ ]:
# Process line by line
json_file = []
for line in content.splitlines():
    # Parse each line if it's valid JSON
    try:
        json_line = json.loads(line)
        json_file.append(json_line)
    except json.JSONDecodeError:
        print("Invalid JSON line:", line)

In [ ]:
df_sitios = pd.DataFrame(json_file)

In [ ]:
df_sitios_no_duplicates = df_sitios.drop_duplicates(['name','gmap_id','latitude','longitude','address'])

In [ ]:
df_sitios_no_duplicates.columns

Index(['name', 'address', 'gmap_id', 'description', 'latitude', 'longitude',
       'category', 'avg_rating', 'num_of_reviews', 'hours', 'MISC', 'state'],
      dtype='object')

In [ ]:
df_sitios_no_duplicates = df_sitios_no_duplicates.drop(columns=['price','relative_results','url'])

In [ ]:
df_sitios_no_duplicates.MISC

0         {'Service options': ['In-store shopping', 'Sam...
1                                                      None
2         {'Service options': ['Takeout', 'Dine-in', 'De...
3         {'Service options': ['In-store shopping'], 'Pa...
4                  {'Service options': ['In-store pickup']}
                                ...                        
274996                                                 None
274997                                                 None
274998    {'Service options': ['Online appointments'], '...
274999    {'Service options': ['Curbside pickup', 'Deliv...
275000    {'Accessibility': ['Wheelchair accessible entr...
Name: MISC, Length: 248428, dtype: object

In [ ]:
import pyarrow.parquet as pq
import gcsfs
import pandas as pd
from pandas_gbq import to_gbq
import bigframes.pandas as bpd
import json
from google.cloud import storage

In [ ]:
    client = storage.Client.from_service_account_json('adsac-455509-0d5538a0d624.json')
    bucket = client.get_bucket('adsac')
    #blob = bucket.blob('Yelp/1.json')
    blob = bucket.blob('Google/1.json')

In [ ]:

    json_file = []
    with blob.open("r", encoding="utf-8") as file_obj:
        for line in file_obj:
            try:
                json_line = json.loads(line.strip())
                json_file.append(json_line)
            except json.JSONDecodeError:
                print("Invalid JSON line:", line.strip())

In [ ]:
    df_sitios = pd.DataFrame(json_file)

In [ ]:
df_sitios.shape

(275001, 15)

In [75]:
my_list = ["apple", "banana", "cherry"]
substring_to_find = "ban"

matching_items = [item for item in my_list if substring_to_find in item]
print(f"Items containing '{substring_to_find}': {matching_items}")


Items containing 'ban': ['banana']


In [ ]:


    df_sitios = df_sitios.drop_duplicates(['name','gmap_id','latitude','longitude','address','address'])
    df_sitios = df_sitios.drop(columns=['price'])

In [90]:

def identify_restaurant(ref):
    #print(ref)
    if not ref:
        return 0
    value =  [item for item in ref if "rest" in item]
    
    if not value:
        return 0
    return 1

In [91]:
df_sitios['is_restaurant'] = df_sitios.category.apply(identify_restaurant)

In [105]:
df_sitios = df_sitios[df_sitios["is_restaurant"] == 1]

In [109]:
    df_sitios = df_sitios.drop_duplicates(['name','address','gmap_id','description','address','avg_rating', 'num_of_reviews'])

In [112]:
    df_sitios['category'] = df_sitios['category'].astype(str)
    df_sitios['hours'] = df_sitios['hours'].astype(str)
    df_sitios['MISC'] = df_sitios['MISC'].astype(str)
    df_sitios['relative_results'] = df_sitios['relative_results'].astype(str)


In [113]:
    #Insertar en la tabla curada
    destination_table = 'adsac-455509.Curated.Sitios'
    # Upload DataFrame
    to_gbq(df_sitios, destination_table, project_id=PROJECT_ID, if_exists='replace')  # Use 'append' if you want to add rows
    print("Termino con éxito la creación de la tabla:",destination_table)

100%|██████████| 1/1 [00:00<?, ?it/s]

Termino con éxito la creación de la tabla: adsac-455509.Curated.Sitios


In [1]:
from functionBigQuery import etl

In [2]:

etl()

100%|██████████| 1/1 [00:00<?, ?it/s]

Termino con éxito la creación de la tabla: adsac-455509.Curated.Tip
